In [10]:
from __future__ import print_function
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data
import torchvision
import torchvision.transforms as transforms
import torchvision.utils as vutils
import numpy as np
import matplotlib.pyplot as plt

import ssl
ssl._create_default_https_context = ssl._create_unverified_context

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
transform = transforms.Compose([
    transforms.Resize(32),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=transform
)
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=128, shuffle=True, num_workers=2
)

In [12]:
from dcgan import Discriminator, Generator

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

G = Generator(ngpu=1).to(device)
D = Discriminator(ngpu=1).to(device)

G.load_state_dict(torch.load('weights/netG_epoch_99.pth', map_location=device))
D.load_state_dict(torch.load('weights/netD_epoch_99.pth', map_location=device))

G.eval()   # G is ALWAYS frozen from here on
D.eval()

print("G and D loaded successfully.")

Running on: cpu
G and D loaded successfully.


In [15]:
# Probe the actual output shape of G
with torch.no_grad():
    z_test = torch.randn(1, 100, 1, 1).to(device)
    out = G(z_test)
    print(f"G output shape: {out.shape}")  # likely (1, 1, 28, 28)

G output shape: torch.Size([1, 1, 28, 28])


In [21]:
class GDagger(nn.Module):
    def __init__(self, ngpu, nc=1, ndf=64, nz=100):
        super(GDagger, self).__init__()
        self.ngpu = ngpu
        self.main = nn.Sequential(
            # (1) x 28 x 28  →  (ndf) x 14 x 14
            nn.Conv2d(nc,      ndf,     3, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # (ndf) x 14 x 14  →  (ndf*2) x 7 x 7
            nn.Conv2d(ndf,     ndf * 2, 3, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),

            # (ndf*2) x 7 x 7  →  (ndf*4) x 4 x 4
            # kernel=4, stride=2, pad=1 on odd dim 7: floor((7+2*1-4)/2)+1 = 4
            nn.Conv2d(ndf * 2, ndf * 4, 3, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),

            # (ndf*4) x 4 x 4  →  (ndf*8) x 1 x 1
            nn.Conv2d(ndf * 4, ndf * 8, 3, 1, 0, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
        )
        # ndf*8 * 1 * 1 = 64*8 = 512
        self.fc = nn.Linear(ndf * 8 * 1 * 1, nz)

    def forward(self, x):
        features = self.main(x)
        features = features.view(features.size(0), -1)  # (B, 512)
        z_hat    = self.fc(features)                     # (B, 100)
        return z_hat.view(-1, 100, 1, 1)

In [22]:
# ── Paper equation (2): dual-loss training ───────────────────
G_dagger = GDagger(ngpu=1).to(device)
G_dagger.train()
G.eval()   # G is ALWAYS frozen

optimizer  = optim.Adam(G_dagger.parameters(), lr=0.0001, betas=(0.5, 0.999))
lambda_reg = 0.1    # from paper: best performance
sigma      = 1.0    # paper: low σ → identity, large σ → breaks idempotence; σ=1 is correct
epochs     = 10000
batch_size = 128

print("Training G† with paper's dual loss...")
for epoch in range(epochs):
    optimizer.zero_grad()

    # Sample z ~ N(0, I_k)
    z_true = torch.randn(batch_size, 100, 1, 1).to(device)

    with torch.no_grad():
        Gz = G(z_true)                                      # G(z)
        nu = torch.randn_like(Gz) * sigma                   # ν ~ N(0, σ²I)
        corrupted = Gz + nu                                  # G(z) + ν

    # G†(G(z) + ν) — encoder pass
    z_recovered = G_dagger(corrupted)

    # G(G†(G(z)+ν)) — re-generate from recovered z
    Gz_reconstructed = G(z_recovered)

    # Paper eq. (2):
    # L = ||G(G†(G(z)+ν)) − G(z)||²  +  λ||G†(G(z)+ν) − z||²
    loss_image   = torch.nn.functional.mse_loss(Gz_reconstructed, Gz)
    loss_latent  = torch.nn.functional.mse_loss(z_recovered, z_true)
    loss         = loss_image + lambda_reg * loss_latent

    loss.backward()
    optimizer.step()

    if epoch % 500 == 0 or epoch == epochs - 1:
        print(f"Epoch [{epoch:>6}/{epochs}] | "
              f"img_loss: {loss_image.item():.6f} | "
              f"z_loss: {loss_latent.item():.6f} | "
              f"total: {loss.item():.6f}")

torch.save(G_dagger.state_dict(), 'weights/G_dagger_correct.pth')
print("Saved.")

Training G† with paper's dual loss...


RuntimeError: mat1 and mat2 shapes cannot be multiplied (128x2048 and 512x100)